# Example 29 - Rigid rotor on anisotropic bearings

This example is based on Example 6.2.3 page 247 from {cite}`friswell2010dynamics`. 

A uniform rigid rotor is shown in figure 3.7. The rotor has length of 0.5 m and a diameter of 0.2 m, and it is made from steel with a density of 7810 kg/m³. The rotor is supported at the ends by bearings. The horizontal and vertical stiffnesses are 1 MN/m and 1.5 MN/m, respectively, at bearing 1 and 1.3 MN/m and 1.8 MN/m, respectively, at bearing 2. The damping values in the horizontal and vertical supports at both bearings are propostional to the stiffnesses and are 20 Ns/m and 30 Ns/m
at bearing 1 and 26 Ns/m and 36 Ns/mat bearing 2. This rotor is essentially the same as that described in Example 3.6.1. Determine the unbalance response over the speed range 0 to 4000 rev/min.

In [1]:
import ross as rs
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np

Q_ = rs.Q_
pio.renderers.default = "notebook_connected"

/home/raphaelts/ross-wt-release-3.2.0/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
steel = rs.Material("steel", E=211e9, G_s=81.2e9, rho=7810)

### Creating rotor

In [3]:
N = 6
L = 0.5 / N
idl = 0
odl = 0.2

shaft = [rs.ShaftElement(L=L, idl=idl, odl=odl, material=steel) for i in range(N)]
bearings = [
    rs.BearingElement(n=0, kxx=1e6, kyy=1.5e6, cxx=20, cyy=30),
    rs.BearingElement(n=6, kxx=1.3e6, kyy=1.8e6, cxx=26, cyy=36),
]
disks = []

rotor = rs.Rotor(shaft_elements=shaft, disk_elements=disks, bearing_elements=bearings)
rotor.plot_rotor()

### Campbell Diagram

In [4]:
fig = rotor.run_campbell(
    speed_range=Q_(list(range(0, 3500, 50)), "RPM"), frequencies=4
).plot()
fig.show()
print(
    "Wn [Hz] at 0 rpm =",
    np.round((rotor.run_modal(speed=0 * np.pi / 30, num_modes=8).wn) / (2 * np.pi), 2),
)

Wn [Hz] at 0 rpm = [21.5  25.93 35.84 42.82]


In [5]:
# Calculating equivalent unbalance mass
e = 0.1e-3
shaft_radius = 0.1
m_unb = (rotor.m * e) / shaft_radius

print(f"Rotor mass = {round(rotor.m, 2)} kg")
print(f"Unbalance mass = {round(m_unb, 2)} kg")

Rotor mass = 122.68 kg
Unbalance mass = 0.12 kg


### Bode Plot

In [6]:
n1 = 3  # position of the unbalance mass in the center of the rotor
m1 = m_unb * (odl / 2)
p1 = 0  # unbalance mass phase
speed = Q_(np.linspace(0, 4000, 60), "RPM")
results_case2 = rotor.run_unbalance_response([n1], [m1], [p1], speed)

/home/raphaelts/ross-wt-release-3.2.0/.venv/lib/python3.14/site-packages/scipy/_lib/_util.py:1138: LinAlgWarning: Diagonal number 39 is exactly zero. Singular matrix.
  return f(*arrays, *other_args, **kwargs)


In [7]:
# Response in x
probe_x = rs.Probe(0, 0)
results_case2.plot(
    probe=[probe_x],
    frequency_units="RPM",
    amplitude_units="µm pkpk",
    phase_units="degrees",
)

In [8]:
# response in y
probe_y = rs.Probe(0, Q_(90, "degrees"))
results_case2.plot(
    probe=[probe_y],
    frequency_units="RPM",
    amplitude_units="µm pkpk",
    phase_units="degrees",
)

In [9]:
# Max response
probe_major = rs.Probe(0, "major")
results_case2.plot(
    probe=[probe_major],
    frequency_units="RPM",
    amplitude_units="µm pkpk",
    phase_units="degrees",
)

### Mode Shapes

In [10]:
speed = 4000 * np.pi / 30
modal = rotor.run_modal(speed)

for i in np.arange(0, 3.1, 1):
    modal.plot_mode_3d(mode=int(i), frequency_units="RPM").show()

In [11]:
for mode in range(4):
    display(modal.plot_orbit(mode, nodes=[0, 3, 6]))

In [12]:
n = [0, 1, 2, 3, 4, 5]  # position of the unbalance mass along the rotor
m1 = m_unb * (odl / 2)
m = [m1 / 6, m1 / 6, m1 / 6, m1 / 6, m1 / 6, m1 / 6]
p = [0, 0, 0, 0, 0, 0]  # unbalance mass phase
speed = Q_(np.linspace(0, 4000, 60), "RPM")
results_case1 = rotor.run_unbalance_response(n, m, p, speed)

probe1 = rs.Probe(0, "major")
results_case1.plot(
    probe=[probe1],
    frequency_units="RPM",
    amplitude_units="µm pkpk",
    phase_units="degrees",
)